# Check the candidate with random games

This notebook validates the **experimental candidate**, separate from notebook 08's active engine. It does not train a model or search for the best deck. It uses implemented cards only; full Standard coverage remains unfinished.

Run the setup cell, then the game-check cell. Defaults: 11 games, up to 500 actions each. A game reaching the limit is unfinished, not a success or failure. Exceptions save a replay file. Even zero exceptions does not prove that all game rules are correct.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
from IPython.display import clear_output

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'tools/stress_candidate.py').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the death-knight-lab project folder.')
CANDIDATE = ROOT / 'staging/rebased-88'
SEEDS_PER_CLASS = 1
MAX_ACTIONS_PER_GAME = 500
CHECK_POLICY_FEATURES = True
# Inspect in a fresh process so notebook 08 imports cannot select the wrong engine.
probe = """import json
from expanded.status import readiness
s=readiness()
s['missing_cards']=len(s['missing_cards'])
print(json.dumps(s))
"""
result = subprocess.run([sys.executable, '-c', probe], cwd=CANDIDATE,
                        capture_output=True, text=True, check=True)
state = json.loads(result.stdout)
print(f"Engine: experimental candidate ({CANDIDATE})")
print(f"Written cards: {state['effects_written']} / {state['catalog_cards']}; missing: {state['missing_cards']}")
print(f"Code fingerprint: {state['fingerprint']}")
receipt = state['fixture_receipt']
if state['fixtures_passed_for_current_code']:
    print(f"Current-code rule checks: {receipt['tests_run']} passed ({receipt['completed_at']})")
else:
    print('Current-code rule checks: no matching passing receipt. Saved older results do not validate this code.')
print('Full Standard ready: NO. This run checks implemented candidate cards only.')
for gap in state['remaining']:
    print('Remaining:', gap)
print('Known rule-verification gaps (not exhaustive):')
for gap in state['known_fidelity_gaps']['gaps']:
    print(f"  {gap['id']}: {gap['status']} — {gap['current_behavior']}")
print(f'Ready: {11 * SEEDS_PER_CLASS} games; no training.')


In [ ]:
command = [sys.executable, str(ROOT / 'tools/stress_candidate.py'),
           '--engine-root', str(CANDIDATE), '--seeds', str(SEEDS_PER_CLASS),
           '--max-actions', str(MAX_ACTIONS_PER_GAME)]
if CHECK_POLICY_FEATURES:
    command.append('--check-features')
completed = 0
summary = None
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE,
                      stderr=subprocess.STDOUT, text=True) as process:
    for line in process.stdout:
        try:
            item = json.loads(line)
        except ValueError:
            print(line.rstrip())
            continue
        if 'status' in item:
            completed += 1
            clear_output(wait=True)
            print(f'Checked {completed} / {11 * SEEDS_PER_CLASS} games')
            print(f'Latest game: {item["status"]}; {item["actions"]} actions')
        else:
            summary = item
    return_code = process.wait()

if completed != 11 * SEEDS_PER_CLASS or summary is None:
    raise RuntimeError('The run stopped early. Review the output before trusting an older report.')
report_path = Path(summary['report_path'])
report = json.loads(report_path.read_text())
if report['run_id'] != summary['run_id'] or report['fingerprint'] != summary['fingerprint']:
    raise RuntimeError('Saved report does not match this run.')
print(f'Finished: {report["terminal"]}; capped: {report["capped"]}; errors: {report["errors"]}')
print(f'Report: {report_path}')
print(f"Policy decisions encoded: {report.get('feature_decisions_checked', 0)}; schema: {report.get('feature_schema')}")
for result in report['results']:
    if result['status'] == 'error':
        print(result['error'], result['message'], result['replay_file'])
if return_code:
    print('Some games failed. Keep their replay files for investigation.')
